### Lặp Jacobi (Có đổi dòng tự động)


In [ ]:
import numpy as np
import itertools
from IPython.display import display, Math, Markdown

def matrix_to_latex(M, precision=4):
    if not isinstance(M, np.ndarray): return str(M)
    elif M.ndim == 1:
        inner = " \\\\ ".join([f"{x:.{precision}f}" for x in M])
        return f"\\begin{{bmatrix}} {inner} \\end{{bmatrix}}^T"
    else:
        rows = " \\\\ ".join([" & ".join([f"{x:.{precision}f}" for x in row]) for row in M])
        return f"\\begin{{bmatrix}} {rows} \\end{{bmatrix}}"

def Check_And_Permute_Dominant(A, b):
    n = A.shape[0]
    for perm in itertools.permutations(range(n)):
        A_perm = A[list(perm), :]
        b_perm = b[list(perm)]
        
        is_dominant = True
        for i in range(n):
            if np.abs(A_perm[i, i]) <= np.sum(np.abs(A_perm[i])) - np.abs(A_perm[i, i]):
                is_dominant = False
                break
        if is_dominant:
            return True, A_perm, b_perm, perm
    return False, A, b, None

def Jacobi_Ax_b_Permute(A_input, b_input, x0_input=None, num_iters=None, epsilon=None):
    A_origin = np.array(A_input, dtype=float)
    b_origin = np.array(b_input, dtype=float).flatten()
    n = len(b_origin)
    
    if x0_input is None: x_k = np.zeros(n)
    else: x_k = np.array(x0_input, dtype=float).flatten()
    
    md = []
    md.append("## ❖ LẶP JACOBI (CÓ TỰ ĐỘNG ĐỔI DÒNG TẠO CHÉO TRỘI)")
    md.append("---\n")
    
    md.append("### I. TỰ ĐỘNG KIỂM TRA VÀ ĐỔI DÒNG")
    md.append("Hệ ban đầu:")
    md.append(f"$$ A_{{ban\\_dau}} = {matrix_to_latex(A_origin)} \\quad b_{{ban\\_dau}} = {matrix_to_latex(b_origin)} $$\n")
    
    is_dom, A, b, perm = Check_And_Permute_Dominant(A_origin, b_origin)
    
    if is_dom and perm != tuple(range(n)):
        md.append(f"**✅ ĐÃ ĐỔI DÒNG THÀNH CÔNG!** Thứ tự các hàng mới: {perm}")
        md.append(f"$$ A = {matrix_to_latex(A)} \\quad b = {matrix_to_latex(b)} $$\n")
        md.append("Ma trận A mới đã **chéo trội hàng ngặt**, đảm bảo Jacobi hội tụ.")
    elif is_dom:
        md.append("**✅ Hợp lệ:** Ma trận ban đầu đã chéo trội hàng ngặt sẵn, không cần đổi dòng.")
        A, b = A_origin, b_origin
    else:
        md.append("**⚠️ Cảnh báo:** Đã thử mọi hoán vị nhưng không thể tạo ra ma trận chéo trội hàng ngặt. Thuật toán có thể phân kỳ.")
        A, b = A_origin, b_origin

    # Biến đổi
    D = np.diag(np.diag(A))
    L_plus_U = A - D
    D_inv = np.linalg.inv(D)
    B = -D_inv @ L_plus_U
    d = D_inv @ b
    
    md.append("\n---\n### II. BIẾN ĐỔI x = Bx + d")
    md.append(f"$$ B = -D^{{-1}}(L+U) = {matrix_to_latex(B)} $$")
    md.append(f"$$ d = D^{{-1}}b = {matrix_to_latex(d)} $$\n")
    
    q = np.linalg.norm(B, np.inf)
    md.append(f"- **Chuẩn vô cùng của B:** $q = ||B||_\\infty = {q:.5f}$")
    
    md.append("\n---\n### III. BẢNG QUÁ TRÌNH LẶP JACOBI")
    
    table = ["| $k$ | " + " | ".join([f"$x_{i+1}$" for i in range(n)]) + " | Sai số $||x^{(k)} - x^{(k-1)}||_\\infty$ |",
             "|---|" + "|".join(["---"] * n) + "|---|"]
             
    history = [x_k.copy()]
    k = 0
    while True:
        if k == 0:
            row_str = " | ".join([f"{v:.5f}" for v in x_k])
            table.append(f"| {k} | {row_str} | - |")
        else:
            diff = np.linalg.norm(history[k] - history[k-1], np.inf)
            row_str = " | ".join([f"{v:.5f}" for v in x_k])
            table.append(f"| {k} | {row_str} | {diff:.5f} |")
            
            stop_eps = (epsilon is not None) and (diff <= epsilon or (q/(1-q))*diff <= epsilon if q < 1 else False)
            stop_iter = (num_iters is not None) and (k >= num_iters)
            if stop_eps or stop_iter:
                break
                
        x_next = B @ x_k + d
        history.append(x_next.copy())
        x_k = x_next
        k += 1
        
    md.extend(table)
    
    md.append("\n---\n### IV. KẾT LUẬN")
    md.append(f"Hệ dừng lại tại bước lặp $k = {k}$. Nghiệm gần đúng là:")
    md.append(f"$$ X^{{({k})}} \\approx {matrix_to_latex(x_k, precision=5)} $$")
    
    diff_final = np.linalg.norm(history[-1] - history[-2], np.inf)
    if q < 1:
        sai_so_hau_nghiem = (q / (1 - q)) * diff_final
        md.append(f"**Sai số hậu nghiệm:** $\\frac{{q}}{{1-q}} ||x^{{({k})}} - x^{{({k-1})}}||_\\infty = {sai_so_hau_nghiem:.5e}$")
    else:
        md.append(f"**Sai số thực tế:** $||x^{{({k})}} - x^{{({k-1})}}||_\\infty = {diff_final:.5e}$")
        
    display(Markdown('\n'.join(md)))

# ═══════════════════════════════════════════════════════════════════
# NHẬP DỮ LIỆU CỦA BẠN VÀO ĐÂY
# ═══════════════════════════════════════════════════════════════════
A = np.array([
    [ 2.0, 10.0,  1.0],
    [10.0,  1.0,  2.0],
    [ 1.0,  2.0, 10.0]
], dtype=float)

b = np.array([13.0, 13.0, 13.0], dtype=float)
# ═══════════════════════════════════════════════════════════════════

Jacobi_Ax_b_Permute(A, b, num_iters=100, epsilon=1e-6)
